# Análise Completa — Unidades Organizacionais das Subprefeituras de São Paulo

Este notebook é a versão interativa do arquivo `analise_completa_subprefeituras.py`.
A ideia é que você rode uma célula de cada vez (`Shift + Enter`) e veja o
resultado aparecer na hora, em vez de rodar o script inteiro de uma vez e só
ver o resultado final.

O caminho segue 5 passos, na ordem:

1. A pergunta de pesquisa
2. Como a pesquisa foi feita
3. Todas as fontes originais
4. As decisões de análise
5. O código que transforma tudo isso em dado estruturado

Antes de rodar, garanta que o `pandas` está instalado:
```
pip install pandas --break-system-packages
```

In [ ]:
import pandas as pd

# Só pra confirmar, na tela, que a biblioteca carregou certo antes de seguir
print("pandas carregado, versão:", pd.__version__)

: 

## Passo 1 — A pergunta que precisava ser respondida

Antes de qualquer código, existia uma pergunta que **nenhum programa consegue
responder sozinho**, porque a resposta está escrita em texto de lei, não em
nenhum banco de dados público:

> "Dentro de cada uma das 32 subprefeituras de São Paulo, quais setores
> internos existem oficialmente — e com base em qual lei ou decreto cada um
> foi criado?"

Essa pergunta importa pro projeto porque o banco de governança de TIC guarda
informações (bases de dados, riscos) vinculadas a um "setor responsável" — e
esse setor precisa ser um dado real e verificável, não um texto inventado no
cadastro.

**Por que essa pergunta não tem resposta de memória:** a estrutura interna de
qualquer prefeitura pode mudar por decreto a qualquer momento. Responder de
memória (ou "chutar" algo com aparência plausível) seria pior do que não ter
a resposta, porque pareceria confiável sem ser de verdade.

## Passo 2 — Como a pesquisa foi feita

A pesquisa aconteceu em duas etapas técnicas diferentes:

- **(a) Busca:** perguntar a um motor de busca frases relacionadas ao tema, e
  receber de volta uma lista de páginas candidatas — só título e resumo, sem
  ler o conteúdo ainda.
- **(b) Leitura direta ("fetch"):** pedir pro computador trazer o **texto
  completo** de uma página específica já encontrada. É essa etapa que
  efetivamente "lê a lei".

A célula abaixo guarda o histórico de cada busca feita, na ordem em que
aconteceu, com o que cada uma trouxe de útil — dá pra repetir exatamente
essas buscas e conferir por conta própria.

In [2]:
HISTORICO_DE_BUSCAS = [
    {
        "consulta": "SMSUB Secretaria Municipal das Subprefeituras estrutura organizacional organograma São Paulo",
        "o_que_trouxe": "Primeiras páginas oficiais de organograma da SMSUB e de subprefeituras individuais.",
    },
    {
        "consulta": "Decreto 65.089 2026 SMSUB texto integral legislação prefeitura São Paulo",
        "o_que_trouxe": "Confirmação da existência e da data do Decreto nº 65.089/2026 (reorganização da SMSUB CENTRAL, não das subprefeituras).",
    },
    {
        "consulta": "decreto 65.089 10 abril 2026 Art. 3º SMSUB coordenadoria texto completo",
        "o_que_trouxe": "Texto literal dos artigos 3º a 8º do Decreto 65.089/2026 (usado para validar a estrutura da SMSUB central, um trabalho anterior a este notebook).",
    },
    {
        "consulta": "Lei 13.682/2003 estrutura administrativa subprefeituras São Paulo departamentos",
        "o_que_trouxe": "Localização da Lei nº 13.682/2003 como a norma-base da estrutura das 32 subprefeituras, e menção aos Decretos nº 42.237/2002 e 42.239/2002.",
    },
    {
        "consulta": "Lei 13682 2003 artigo 2 estrutura organizacional subprefeituras Coordenadoria Supervisão",
        "o_que_trouxe": "Trechos do Artigo 1º da Lei 13.682/2003 listando Gabinete e Coordenadorias; confirmação de que a Coordenadoria de Governo Local (CGL) NÃO está nessa lei — veio depois.",
    },
    {
        "consulta": "(pesquisa aprofundada, com dezenas de fontes cruzadas automaticamente)",
        "o_que_trouxe": (
            "Confirmação de que a estrutura é PADRONIZADA nas 32 subprefeituras; "
            "localização do Decreto nº 57.588/2017 como a norma que criou a "
            "Coordenadoria de Governo Local (CGL); confirmação de que NÃO existe "
            "unidade formal de TIC dentro das subprefeituras individuais (só um "
            "núcleo informal, 'Informática/AJTI', sem lei própria)."
        ),
    },
]

# Mostrando na tela, uma busca por linha, pra você ver o histórico completo
for item in HISTORICO_DE_BUSCAS:
    print("BUSCA:", item["consulta"])
    print("  ->", item["o_que_trouxe"])
    print()

BUSCA: SMSUB Secretaria Municipal das Subprefeituras estrutura organizacional organograma São Paulo
  -> Primeiras páginas oficiais de organograma da SMSUB e de subprefeituras individuais.

BUSCA: Decreto 65.089 2026 SMSUB texto integral legislação prefeitura São Paulo
  -> Confirmação da existência e da data do Decreto nº 65.089/2026 (reorganização da SMSUB CENTRAL, não das subprefeituras).

BUSCA: decreto 65.089 10 abril 2026 Art. 3º SMSUB coordenadoria texto completo
  -> Texto literal dos artigos 3º a 8º do Decreto 65.089/2026 (usado para validar a estrutura da SMSUB central, um trabalho anterior a este notebook).

BUSCA: Lei 13.682/2003 estrutura administrativa subprefeituras São Paulo departamentos
  -> Localização da Lei nº 13.682/2003 como a norma-base da estrutura das 32 subprefeituras, e menção aos Decretos nº 42.237/2002 e 42.239/2002.

BUSCA: Lei 13682 2003 artigo 2 estrutura organizacional subprefeituras Coordenadoria Supervisão
  -> Trechos do Artigo 1º da Lei 13.682/2003

## Passo 3 — Todas as fontes originais, guardadas aqui dentro

Esta é a parte que resolve o problema de "não tem como comprovar de onde
tirei": cada fonte usada neste notebook está listada abaixo, com o link
exato. Nenhuma fonte usada no Passo 5 (o molde de unidades) fica de fora
desta lista — inclusive as que **não** foram confirmadas em texto completo
(marcadas com link `None`, de propósito, em vez de escondidas).

In [3]:
FONTES = {
    # --- Leis e decretos (as normas que efetivamente criam cada unidade) ---
    "lei_13399_2002": {
        "titulo": "Lei nº 13.399, de 1º de agosto de 2002",
        "descricao": "Cria as subprefeituras no Município de São Paulo.",
        "url": "https://plpconsulta.saopaulo.sp.leg.br/Forms/MostrarArquivo?TIPO=Lei&NUMERO=13399&ANO=2002&DOCUMENTO=Atualizado",
    },
    "lei_13682_2003": {
        "titulo": "Lei nº 13.682, de 15 de dezembro de 2003",
        "descricao": "Estabelece a estrutura organizacional das Subprefeituras — a norma-base usada para Gabinete, CAF, CPDU e CPO neste notebook.",
        "url": "https://legislacao.prefeitura.sp.gov.br/leis/lei-13682-de-15-de-dezembro-de-2003",
    },
    "decreto_57588_2017": {
        "titulo": "Decreto nº 57.588, de 10 de fevereiro de 2017",
        "descricao": "Cria a Coordenadoria de Governo Local (CGL), reunindo Habitação, Cultura e Esportes/Lazer.",
        "url": "https://legislacao.prefeitura.sp.gov.br/leis/decreto-57588-de-10-de-fevereiro-de-2017",
    },
    "lei_16974_2018": {
        "titulo": "Lei nº 16.974, de 23 de agosto de 2018",
        "descricao": "Reorganiza a Administração Pública Municipal Direta; CONFIRMA (não substitui) a estrutura das subprefeituras e o vínculo delas com a SMSUB.",
        "url": "https://app-plpconsulta-prd.azurewebsites.net/Forms/MostrarArquivo?TIPO=Lei&NUMERO=16974&ANO=2018&DOCUMENTO=Atualizado",
    },
    "decreto_61731_2022": {
        "titulo": "Decreto nº 61.731, de 26 de agosto de 2022",
        "descricao": "Organiza os cargos de provimento em comissão das 32 subprefeituras (referência complementar sobre pessoal, não sobre a estrutura em si).",
        "url": "https://legislacao.prefeitura.sp.gov.br/leis/decreto-61731-de-26-de-agosto-de-2022",
    },

    # --- Organogramas oficiais ---
    "organograma_vila_prudente": {
        "titulo": "Organograma e Estrutura Administrativa — Subprefeitura Vila Prudente",
        "descricao": "O organograma mais completo publicado; referência principal usada para o molde de 22 unidades deste notebook.",
        "url": "https://prefeitura.sp.gov.br/web/vila_prudente/w/acesso_a_informacao/50094",
    },
    "organograma_lapa": {
        "titulo": "Organograma e Estrutura Administrativa — Subprefeitura Lapa",
        "descricao": "Usado para comparar a estrutura com outra subprefeitura de perfil diferente.",
        "url": "https://prefeitura.sp.gov.br/web/lapa/w/acesso_a_informacao/50305",
    },
    "organograma_jacana_tremembe": {
        "titulo": "Organograma e Estrutura Administrativa — Subprefeitura Jaçanã/Tremembé",
        "descricao": "Usado para comparar a estrutura com uma subprefeitura de porte médio/periférico.",
        "url": "https://prefeitura.sp.gov.br/web/jacana_tremembe/w/acesso_a_informacao/50283-1",
    },
    "organograma_casa_verde": {
        "titulo": "Organograma e Estrutura Administrativa — Subprefeitura Casa Verde",
        "descricao": "Usado para comparar a estrutura com mais uma subprefeitura de perfil diferente.",
        "url": "https://prefeitura.sp.gov.br/web/casa_verde/w/acesso_a_informacao/50009",
    },
    "competencias_se": {
        "titulo": "Competências e Atribuições — Subprefeitura Sé",
        "descricao": "Usado para comparar a estrutura com uma subprefeitura central.",
        "url": "https://prefeitura.sp.gov.br/web/se/w/acesso_a_informacao/50191",
    },
    "competencias_subprefeituras_geral": {
        "titulo": "Competências e Atribuições — Subprefeituras (página geral da SMSUB)",
        "descricao": "Página institucional geral, usada para localizar a lista de normas relacionadas.",
        "url": "https://prefeitura.sp.gov.br/web/subprefeituras/w/acesso_a_informacao/178392",
    },
    "orientacoes_tecnicas_tic": {
        "titulo": "Orientações Técnicas de TIC nas subprefeituras",
        "descricao": "Confirma que o suporte de TIC às subprefeituras é orientado de forma centralizada, sem unidade formal de TIC local.",
        "url": "https://tecnologia.prefeitura.sp.gov.br/?p=3212",
    },

    # --- Fontes mencionadas, mas cujo texto completo NÃO foi recuperado ---
    "decreto_42237_2002_nao_confirmado": {
        "titulo": "Decreto nº 42.237/2002 (referência não verificada em texto completo)",
        "descricao": "Citado em páginas oficiais de subprefeitura como parte da base de implantação. Texto integral não recuperado nesta pesquisa.",
        "url": None,
    },
    "decreto_42239_2002_nao_confirmado": {
        "titulo": "Decreto nº 42.239/2002 (referência não verificada em texto completo)",
        "descricao": "Citado em páginas oficiais de subprefeitura como parte da base de implantação. Texto integral não recuperado nesta pesquisa.",
        "url": None,
    },
}

print(f"Total de fontes registradas: {len(FONTES)}")
print(f"Fontes com link confirmado: {sum(1 for f in FONTES.values() if f['url'])}")
print(f"Fontes sem confirmação de texto completo: {sum(1 for f in FONTES.values() if not f['url'])}")

Total de fontes registradas: 14
Fontes com link confirmado: 12
Fontes sem confirmação de texto completo: 2


In [4]:
# Ver a bibliografia inteira formatada, uma fonte por vez
for chave, dado in FONTES.items():
    print(f"- {dado['titulo']}")
    print(f"  {dado['descricao']}")
    print(f"  Link: {dado['url'] if dado['url'] else '(texto completo não recuperado nesta pesquisa)'}")
    print()

- Lei nº 13.399, de 1º de agosto de 2002
  Cria as subprefeituras no Município de São Paulo.
  Link: https://plpconsulta.saopaulo.sp.leg.br/Forms/MostrarArquivo?TIPO=Lei&NUMERO=13399&ANO=2002&DOCUMENTO=Atualizado

- Lei nº 13.682, de 15 de dezembro de 2003
  Estabelece a estrutura organizacional das Subprefeituras — a norma-base usada para Gabinete, CAF, CPDU e CPO neste notebook.
  Link: https://legislacao.prefeitura.sp.gov.br/leis/lei-13682-de-15-de-dezembro-de-2003

- Decreto nº 57.588, de 10 de fevereiro de 2017
  Cria a Coordenadoria de Governo Local (CGL), reunindo Habitação, Cultura e Esportes/Lazer.
  Link: https://legislacao.prefeitura.sp.gov.br/leis/decreto-57588-de-10-de-fevereiro-de-2017

- Lei nº 16.974, de 23 de agosto de 2018
  Reorganiza a Administração Pública Municipal Direta; CONFIRMA (não substitui) a estrutura das subprefeituras e o vínculo delas com a SMSUB.
  Link: https://app-plpconsulta-prd.azurewebsites.net/Forms/MostrarArquivo?TIPO=Lei&NUMERO=16974&ANO=2018&D

## Passo 4 — As decisões de análise (não é cópia, é interpretação)

Ler uma lei e copiar um nome de coordenadoria para dentro de uma lista não é,
sozinho, "análise". A análise de verdade está nas **decisões** que precisei
tomar, comparando informações de fontes diferentes.

### Decisão 1 — "A estrutura é igual nas 32 subprefeituras, ou cada uma tem a sua própria?"

Comparei os organogramas de Vila Prudente, Lapa, Jaçanã/Tremembé, Casa Verde
e Sé — de propósito, escolhendo perfis bem diferentes entre si (central,
periférica, de porte médio). Resultado: todas seguem o **mesmo desenho**
(Gabinete + CAF + CPDU + CPO + CGL). As diferenças encontradas eram só de
apelido de exibição numa página específica, não de estrutura.

**Consequência prática:** por isso o código do Passo 5 usa um único "molde",
repetido para as 32, em vez de 32 listas diferentes.

### Decisão 2 — "Quem pertence a quem, dentro da estrutura?"

A lei não desenha uma árvore — ela lista as unidades em sequência de artigos
e incisos. Fui eu que precisei interpretar, por exemplo, que a "Supervisão de
Administração e Suprimentos" é subordinada à "Coordenadoria de Administração
e Finanças", e não à subprefeitura diretamente.

### Decisão 3 — "O que fazer com o que a pesquisa NÃO encontrou?"

O caso mais importante: o "Núcleo de Informática" aparece nos organogramas
administrativos, mas **nenhuma lei ou decreto pesquisado cria essa unidade
formalmente**. A decisão foi registrar esse fato dentro do próprio dado
(campo `norma` = "Sem base legal formal — núcleo administrativo"), em vez de
inventar uma referência legal falsa só para preencher o campo.

## Passo 5 — O código: transformando as decisões acima em dado estruturado

A partir daqui é o pipeline de dados de verdade. Cada célula é um pedacinho
— rode uma de cada vez e veja o resultado antes de ir pra próxima.

### 5.1 — As 32 subprefeituras

O código de cada uma, igual ao que já existe no `init_subprefeituras.cypher`.

In [5]:
SUBPREFEITURAS = [
    "AD", "AF", "BT", "CL", "CS", "CT", "CV", "EM", "FB", "G", "IP", "IQ",
    "IT", "JA", "JT", "LA", "MB", "MG", "MO", "MP", "PA", "PE", "PI", "PJ",
    "PR", "SA", "SB", "SE", "SM", "ST", "VM", "VP",
]

print(f"Total de subprefeituras: {len(SUBPREFEITURAS)}")
print(SUBPREFEITURAS)

Total de subprefeituras: 32
['AD', 'AF', 'BT', 'CL', 'CS', 'CT', 'CV', 'EM', 'FB', 'G', 'IP', 'IQ', 'IT', 'JA', 'JT', 'LA', 'MB', 'MG', 'MO', 'MP', 'PA', 'PE', 'PI', 'PJ', 'PR', 'SA', 'SB', 'SE', 'SM', 'ST', 'VM', 'VP']


### 5.2 — O molde de 22 unidades (o "template")

Cada dicionário é uma unidade. Campo por campo:

- `sufixo` — o pedaço da sigla que vai ser prefixado com o código da subprefeitura (vira, por ex., `AD-CAF`)
- `nome` — o nome oficial, como consta na norma
- `tipo` — o nível hierárquico
- `norma` / `artigo` — a base legal (ligação com o Passo 3, via `FONTES`)
- `pertence_a_no_molde` — `None` = pendura direto na subprefeitura; um sufixo = pendura dentro daquela outra unidade (aqui a Decisão 2 vira código)
- `fonte_url` — o link direto pra prova de cada linha (aqui a Decisão 3 também aparece: quando não existe lei, o link aponta pro organograma)

In [6]:
TEMPLATE_UNIDADES = [
    # -- Gabinete do Subprefeito e suas assessorias diretas --
    {"sufixo": "GAB",    "nome": "Gabinete do Subprefeito",                         "tipo": "Gabinete",          "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, I",  "pertence_a_no_molde": None, "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "AJ",     "nome": "Assessoria Jurídica",                              "tipo": "Assessoria",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, I",  "pertence_a_no_molde": "GAB", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "AC",     "nome": "Assessoria Executiva de Comunicação",              "tipo": "Assessoria",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, I",  "pertence_a_no_molde": "GAB", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "AT",     "nome": "Assessoria Técnica",                                "tipo": "Assessoria",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, I",  "pertence_a_no_molde": "GAB", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "PAT",    "nome": "Praça de Atendimento ao Público",                  "tipo": "Unidade",           "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, I",  "pertence_a_no_molde": "GAB", "fonte_url": FONTES["lei_13682_2003"]["url"]},

    # -- Coordenadoria de Administração e Finanças --
    {"sufixo": "CAF",    "nome": "Coordenadoria de Administração e Finanças",        "tipo": "Coordenadoria",     "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, VIII", "pertence_a_no_molde": None, "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "SAS",    "nome": "Supervisão de Administração e Suprimentos",        "tipo": "Supervisão",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, VIII", "pertence_a_no_molde": "CAF", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "SF",     "nome": "Supervisão de Finanças",                           "tipo": "Supervisão",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, VIII", "pertence_a_no_molde": "CAF", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "SUGESP", "nome": "Supervisão de Gestão de Pessoas",                  "tipo": "Supervisão",        "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, VIII", "pertence_a_no_molde": "CAF", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "AJTI",   "nome": "Núcleo de Informática",                            "tipo": "Núcleo (informal)", "norma": "Sem base legal formal — núcleo administrativo", "artigo": None, "pertence_a_no_molde": "CAF", "fonte_url": FONTES["organograma_vila_prudente"]["url"]},

    # -- Coordenadoria de Planejamento e Desenvolvimento Urbano --
    {"sufixo": "CPDU",   "nome": "Coordenadoria de Planejamento e Desenvolvimento Urbano", "tipo": "Coordenadoria", "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, III", "pertence_a_no_molde": None, "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STUSL",  "nome": "Supervisão Técnica de Uso do Solo e Licenciamentos", "tipo": "Supervisão",      "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, III", "pertence_a_no_molde": "CPDU", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STPU",   "nome": "Supervisão Técnica de Planejamento Urbano",         "tipo": "Supervisão",       "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, III", "pertence_a_no_molde": "CPDU", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STF",    "nome": "Supervisão Técnica de Fiscalização",                "tipo": "Supervisão",       "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, III", "pertence_a_no_molde": "CPDU", "fonte_url": FONTES["lei_13682_2003"]["url"]},

    # -- Coordenadoria de Projetos e Obras --
    {"sufixo": "CPO",    "nome": "Coordenadoria de Projetos e Obras",                 "tipo": "Coordenadoria",    "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, IV e V", "pertence_a_no_molde": None, "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STPO",   "nome": "Supervisão Técnica de Projetos e Obras",            "tipo": "Supervisão",       "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, IV e V", "pertence_a_no_molde": "CPO", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STLP",   "nome": "Supervisão Técnica de Limpeza Pública",             "tipo": "Supervisão",       "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, IV e V", "pertence_a_no_molde": "CPO", "fonte_url": FONTES["lei_13682_2003"]["url"]},
    {"sufixo": "STM",    "nome": "Supervisão Técnica de Manutenção",                  "tipo": "Supervisão",       "norma": "Lei nº 13.682/2003", "artigo": "Art. 1º, IV e V", "pertence_a_no_molde": "CPO", "fonte_url": FONTES["lei_13682_2003"]["url"]},

    # -- Coordenadoria de Governo Local (a mais recente, criada em 2017) --
    {"sufixo": "CGL",    "nome": "Coordenadoria de Governo Local",                    "tipo": "Coordenadoria",    "norma": "Decreto nº 57.588/2017", "artigo": "Art. 1º", "pertence_a_no_molde": None, "fonte_url": FONTES["decreto_57588_2017"]["url"]},
    {"sufixo": "HABI",   "nome": "Supervisão de Habitação",                           "tipo": "Supervisão",       "norma": "Decreto nº 57.588/2017", "artigo": "Art. 2º", "pertence_a_no_molde": "CGL", "fonte_url": FONTES["decreto_57588_2017"]["url"]},
    {"sufixo": "SC",     "nome": "Supervisão de Cultura",                             "tipo": "Supervisão",       "norma": "Decreto nº 57.588/2017", "artigo": "Art. 2º", "pertence_a_no_molde": "CGL", "fonte_url": FONTES["decreto_57588_2017"]["url"]},
    {"sufixo": "SEL",    "nome": "Supervisão de Esportes e Lazer",                    "tipo": "Supervisão",       "norma": "Decreto nº 57.588/2017", "artigo": "Art. 2º", "pertence_a_no_molde": "CGL", "fonte_url": FONTES["decreto_57588_2017"]["url"]},
]

print(f"Total de unidades no molde (por subprefeitura): {len(TEMPLATE_UNIDADES)}")

Total de unidades no molde (por subprefeitura): 22


### 5.3 — Ver o molde como tabela

Aqui já dá pra visualizar: vire o molde num DataFrame do pandas (uma
"planilha" dentro do Python) e mostre ele na tela.

In [7]:
df_template = pd.DataFrame(TEMPLATE_UNIDADES)
df_template  # em uma célula de notebook, escrever o nome da variável já mostra a tabela

,sufixo,nome,tipo,norma,artigo,pertence_a_no_molde,fonte_url
0,GAB,Gabinete do Subprefeito,Gabinete,Lei nº 13.682/2003,"Art. 1º, I",NaN,https://legislacao.prefeitura.sp.gov.br/leis/l...
1,AJ,Assessoria Jurídica,Assessoria,Lei nº 13.682/2003,"Art. 1º, I",GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
2,AC,Assessoria Executiva de Comunicação,Assessoria,Lei nº 13.682/2003,"Art. 1º, I",GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
3,AT,Assessoria Técnica,Assessoria,Lei nº 13.682/2003,"Art. 1º, I",GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
4,PAT,Praça de Atendimento ao Público,Unidade,Lei nº 13.682/2003,"Art. 1º, I",GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
5,CAF,Coordenadoria de Administração e Finanças,Coordenadoria,Lei nº 13.682/2003,"Art. 1º, VIII",NaN,https://legislacao.prefeitura.sp.gov.br/leis/l...
6,SAS,Supervisão de Administração e Suprimentos,Supervisão,Lei nº 13.682/2003,"Art. 1º, VIII",CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
7,SF,Supervisão de Finanças,Supervisão,Lei nº 13.682/2003,"Art. 1º, VIII",CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
8,SUGESP,Supervisão de Gestão de Pessoas,Supervisão,Lei nº 13.682/2003,"Art. 1º, VIII",CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
9,AJTI,Núcleo de Informática,Núcleo (informal),Sem base legal formal — núcleo administrativo,NaN,CAF,https://prefeitura.sp.gov.br/web/vila_prudente...


### 5.4 — Expandir o molde para as 32 subprefeituras

Aqui é onde a "análise" de verdade acontece de forma automática: cruzar
(produto cartesiano) cada uma das 22 unidades do molde com cada uma das 32
subprefeituras, gerando as 704 linhas finais.

In [8]:
def expandir_para_todas_subprefeituras(template: pd.DataFrame, subprefeituras: list[str]) -> pd.DataFrame:
    linhas_finais = []

    for sub in subprefeituras:
        nome_orgao_pai = f"SUB-{sub}"

        for _, unidade in template.iterrows():
            sigla_completa = f"{sub}-{unidade['sufixo']}"

            # Pegadinha do pandas: None dentro de uma coluna mista vira NaN.
            # "is None" NÃO reconhece NaN. pd.isna() reconhece os dois.
            if pd.isna(unidade["pertence_a_no_molde"]):
                pertence_a = nome_orgao_pai
            else:
                pertence_a = f"{sub}-{unidade['pertence_a_no_molde']}"

            linhas_finais.append({
                "norma": unidade["norma"],
                "artigo": unidade["artigo"],
                "sigla": sigla_completa,
                "nome": unidade["nome"],
                "tipo": unidade["tipo"],
                "pertence_a": pertence_a,
                "fonte_url": unidade["fonte_url"],
            })

    return pd.DataFrame(linhas_finais)


df_completo = expandir_para_todas_subprefeituras(df_template, SUBPREFEITURAS)
print(f"Total de linhas geradas: {len(df_completo)}")
df_completo.head(10)  # mostra só as 10 primeiras, pra não poluir a tela

Total de linhas geradas: 704


,norma,artigo,sigla,nome,tipo,pertence_a,fonte_url
0,Lei nº 13.682/2003,"Art. 1º, I",AD-GAB,Gabinete do Subprefeito,Gabinete,SUB-AD,https://legislacao.prefeitura.sp.gov.br/leis/l...
1,Lei nº 13.682/2003,"Art. 1º, I",AD-AJ,Assessoria Jurídica,Assessoria,AD-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
2,Lei nº 13.682/2003,"Art. 1º, I",AD-AC,Assessoria Executiva de Comunicação,Assessoria,AD-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
3,Lei nº 13.682/2003,"Art. 1º, I",AD-AT,Assessoria Técnica,Assessoria,AD-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
4,Lei nº 13.682/2003,"Art. 1º, I",AD-PAT,Praça de Atendimento ao Público,Unidade,AD-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
5,Lei nº 13.682/2003,"Art. 1º, VIII",AD-CAF,Coordenadoria de Administração e Finanças,Coordenadoria,SUB-AD,https://legislacao.prefeitura.sp.gov.br/leis/l...
6,Lei nº 13.682/2003,"Art. 1º, VIII",AD-SAS,Supervisão de Administração e Suprimentos,Supervisão,AD-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
7,Lei nº 13.682/2003,"Art. 1º, VIII",AD-SF,Supervisão de Finanças,Supervisão,AD-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
8,Lei nº 13.682/2003,"Art. 1º, VIII",AD-SUGESP,Supervisão de Gestão de Pessoas,Supervisão,AD-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
9,Sem base legal formal — núcleo administrativo,NaN,AD-AJTI,Núcleo de Informática,Núcleo (informal),AD-CAF,https://prefeitura.sp.gov.br/web/vila_prudente...


### 5.5 — Conferir os dados de uma subprefeitura específica

Filtro rápido, só pra "ver de perto" — troque `"BT"` por qualquer código de
subprefeitura da lista acima.

In [9]:
df_completo[df_completo["sigla"].str.startswith("BT-")]

,norma,artigo,sigla,nome,tipo,pertence_a,fonte_url
44,Lei nº 13.682/2003,"Art. 1º, I",BT-GAB,Gabinete do Subprefeito,Gabinete,SUB-BT,https://legislacao.prefeitura.sp.gov.br/leis/l...
45,Lei nº 13.682/2003,"Art. 1º, I",BT-AJ,Assessoria Jurídica,Assessoria,BT-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
46,Lei nº 13.682/2003,"Art. 1º, I",BT-AC,Assessoria Executiva de Comunicação,Assessoria,BT-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
47,Lei nº 13.682/2003,"Art. 1º, I",BT-AT,Assessoria Técnica,Assessoria,BT-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
48,Lei nº 13.682/2003,"Art. 1º, I",BT-PAT,Praça de Atendimento ao Público,Unidade,BT-GAB,https://legislacao.prefeitura.sp.gov.br/leis/l...
49,Lei nº 13.682/2003,"Art. 1º, VIII",BT-CAF,Coordenadoria de Administração e Finanças,Coordenadoria,SUB-BT,https://legislacao.prefeitura.sp.gov.br/leis/l...
50,Lei nº 13.682/2003,"Art. 1º, VIII",BT-SAS,Supervisão de Administração e Suprimentos,Supervisão,BT-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
51,Lei nº 13.682/2003,"Art. 1º, VIII",BT-SF,Supervisão de Finanças,Supervisão,BT-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
52,Lei nº 13.682/2003,"Art. 1º, VIII",BT-SUGESP,Supervisão de Gestão de Pessoas,Supervisão,BT-CAF,https://legislacao.prefeitura.sp.gov.br/leis/l...
53,Sem base legal formal — núcleo administrativo,NaN,BT-AJTI,Núcleo de Informática,Núcleo (informal),BT-CAF,https://prefeitura.sp.gov.br/web/vila_prudente...


### 5.6 — Análise e validação (a auditoria automática)

Antes de gravar o CSV, essa etapa confere se os dados fazem sentido —
contagens, e checagens automáticas que pegariam um erro na hora.

In [10]:
def analisar_e_validar(df: pd.DataFrame) -> None:
    print(f"Total de linhas: {len(df)}")
    print(f"Total de subprefeituras representadas: {df['sigla'].str.slice(0, 2).nunique()}")
    print()

    print("Quantidade de unidades por tipo:")
    print(df["tipo"].value_counts().to_string())
    print()

    print("Quantidade de unidades por norma:")
    print(df["norma"].value_counts().to_string())
    print()

    duplicadas = df[df.duplicated(subset="sigla", keep=False)]
    print("[OK] Nenhuma sigla duplicada." if duplicadas.empty else f"[ATENÇÃO] {len(duplicadas)} siglas duplicadas.")

    siglas_existentes = set(df["sigla"])
    aponta_para_unidade = df[~df["pertence_a"].str.startswith("SUB-")]
    orfaos = aponta_para_unidade[~aponta_para_unidade["pertence_a"].isin(siglas_existentes)]
    print("[OK] Todo 'pertence_a' aponta para algo que existe." if orfaos.empty else f"[ATENÇÃO] {len(orfaos)} linhas órfãs.")

    sem_fonte = df[df["fonte_url"].isna() | (df["fonte_url"].str.strip() == "")]
    print("[OK] Todas as linhas têm fonte_url preenchida." if sem_fonte.empty else f"[ATENÇÃO] {len(sem_fonte)} linhas sem fonte.")


analisar_e_validar(df_completo)

Total de linhas: 704
Total de subprefeituras representadas: 32

Quantidade de unidades por tipo:
tipo
Supervisão           384
Coordenadoria        128
Assessoria            96
Gabinete              32
Unidade               32
Núcleo (informal)     32

Quantidade de unidades por norma:
norma
Lei nº 13.682/2003                               544
Decreto nº 57.588/2017                           128
Sem base legal formal — núcleo administrativo     32

[OK] Nenhuma sigla duplicada.


[OK] Todo 'pertence_a' aponta para algo que existe.
[OK] Todas as linhas têm fonte_url preenchida.


### 5.7 — Gerar o arquivo CSV final

Por fim, salva o resultado — o mesmo `subprefeituras_unidades.csv` usado
pelo `init_subprefeituras_unidades.cypher`.

In [11]:
caminho_saida = "subprefeituras_unidades.csv"
df_completo.to_csv(caminho_saida, index=False, encoding="utf-8")
print(f"Arquivo gerado: {caminho_saida}")
print(f"Linhas de dados: {len(df_completo)} (+ 1 linha de cabeçalho)")

Arquivo gerado: subprefeituras_unidades.csv
Linhas de dados: 704 (+ 1 linha de cabeçalho)
